# MSA → Egyptian & Levantine Dialect Translation

This notebook translates an Arabic Boundary-Set JSON file from **MSA** into **Egyptian** and **Levantine** dialects using the QCRI AraDiCE models.

## How to use

1. Make sure the runtime has a GPU: **Runtime → Change runtime type → T4 GPU**.
2. Upload your input file `Arabic Boundary-set_MSA.json` into `/content/` (drag-and-drop into the file panel).
3. **Run all cells** (`Runtime → Run all`).
4. The outputs `Arabic boundary-set_EGY.json` and `Arabic boundary-set_LEV.json` will appear in `/content/`.

The notebook runs both models **sequentially on a single GPU** (Colab gives you one), uses **FP16** to keep memory low, and frees GPU memory between models.


## 1. Install dependencies

In [ ]:
!pip install -q --upgrade "transformers>=4.41" sentencepiece tqdm

## 2. Verify GPU

In [ ]:
import torch
print("CUDA available :", torch.cuda.is_available())
print("GPU count      :", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name       :", torch.cuda.get_device_name(0))
    print("VRAM (GB)      :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    raise RuntimeError("No GPU detected. Go to Runtime > Change runtime type and pick a GPU.")

## 3. Configuration

Both models use `cuda:0` because Colab provides one GPU. They run sequentially with memory freed in between.

In [ ]:
import json
import gc
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm.auto import tqdm

INPUT_FILE = "/content/Arabic Boundary-set_MSA.json"
BATCH_SIZE = 32
MAX_LENGTH = 512

MODELS = {
    "EGY": {
        "name": "QCRI/AraDiCE-msa-to-egy",
        "bos_token": "arz_Arab",
        "gpu": 0,
        "output_file": "/content/Arabic boundary-set_EGY.json",
    },
    "LEV": {
        "name": "QCRI/AraDiCE-msa-to-lev",
        "bos_token": "ajp_Arab",
        "gpu": 0,
        "output_file": "/content/Arabic boundary-set_LEV.json",
    },
}

## 4. Translation functions

Same logic as the original script, with FP16 added for speed and memory.

In [ ]:
def load_model(model_cfg):
    device = f"cuda:{model_cfg['gpu']}"
    print(f"Loading {model_cfg['name']} on {device}...")
    tokenizer = AutoTokenizer.from_pretrained(model_cfg["name"])
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_cfg["name"],
        torch_dtype=torch.float16,   # half precision — ~2x faster, half the VRAM
    ).to(device)
    model.eval()
    forced_bos_id = tokenizer.convert_tokens_to_ids(model_cfg["bos_token"])
    print(f"Loaded {model_cfg['name']} on {device}")
    return tokenizer, model, device, forced_bos_id


def translate_batch(prompts, tokenizer, model, device, forced_bos_id):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(device)
    with torch.no_grad():
        translated = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_id,
            max_length=MAX_LENGTH,
            num_beams=1,             # greedy decoding for speed
        )
    return tokenizer.batch_decode(translated, skip_special_tokens=True)


def translate_dialect(data, model_cfg, loaded_model):
    tokenizer, model, device, forced_bos_id = loaded_model
    prompts = [item["Prompt"] for item in data]
    translations = []

    for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc=f"{model_cfg['name']}"):
        batch = prompts[i : i + BATCH_SIZE]
        translated = translate_batch(batch, tokenizer, model, device, forced_bos_id)
        translations.extend(translated)

    result = []
    for item, translated_prompt in zip(data, translations):
        result.append({
            "Safe/Unsafe": item["Safe/Unsafe"],
            "Category":    item["Category"],
            "Prompt":      translated_prompt,
        })

    with open(model_cfg["output_file"], "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"Saved {model_cfg['output_file']}")

## 5. Run translation

This is the slow cell. Each model is downloaded (~13 GB the first time), loaded, used, then unloaded before the next one starts. Expect roughly **15–20 minutes** end-to-end on a T4.

In [ ]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)
print(f"Loaded {len(data)} prompts from {INPUT_FILE}")

for dialect, cfg in MODELS.items():
    print(f"\n=== {dialect} ===")
    loaded = load_model(cfg)
    translate_dialect(data, cfg, loaded)

    # Free GPU memory before loading the next model
    tokenizer, model, device, forced_bos_id = loaded
    del tokenizer, model, loaded
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Done: {dialect}")

print("\nAll translations complete!")

## 6. Preview the outputs

In [ ]:
import json

for cfg in MODELS.values():
    with open(cfg["output_file"], "r", encoding="utf-8") as f:
        out = json.load(f)
    print(f"\n=== {cfg['output_file']} ({len(out)} entries) ===")
    for entry in out[:3]:
        print(json.dumps(entry, ensure_ascii=False, indent=2))

---

The output files are now at:

- `/content/Arabic boundary-set_EGY.json`
- `/content/Arabic boundary-set_LEV.json`

Right-click each one in the file panel to download.
